In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\USER\miniconda3\envs\env_RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
OPENROUTER_API_KEY = "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"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


In [10]:
loader = WebBaseLoader("https://cognimobility.com/about-us")
documents = loader.load()


In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

vector_db = FAISS.from_documents(docs, embeddings)
retriever = vector_db.as_retriever()

In [12]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0
)

In [13]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""
)


In [14]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print(rag_chain.invoke("Who is Ahmand Mustafa?"))


Engr. Ahmad Mustafa is a researcher and technology enthusiast specializing in Intelligent Transportation Systems (ITS), AI, and Smart Mobility. He focuses on traffic flow forecasting, sensor fusion, and deep learning to build innovative solutions for autonomous vehicles and smart cities. He has experience as a Lab Engineer in Computer Systems Engineering, contributions to IoT-based sensor development, and holds a patent in ITS.
